# **Main Pipeline - Waste Classification**

#### *This notebook runs the complete project pipeline: download, dataset split, preprocessing, model building, training, evaluation, best-model comparison, and prediction.*


## 1. Project initialization

In [14]:
from pathlib import Path
import os
import sys
import importlib
import json

from IPython.display import display, Image as IPyImage

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

os.chdir(PROJECT_ROOT)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project directory :", PROJECT_ROOT)
print("Source directory    :", SRC_DIR)


Project directory : C:\Users\Trump\OneDrive - USherbrooke\Documents\Projets_github\Images_classification_RN50_CNN
Source directory    : C:\Users\Trump\OneDrive - USherbrooke\Documents\Projets_github\Images_classification_RN50_CNN\src


## 2. Module imports

In [15]:
import config
import download_dataset
import split_dataset
import data_preprocessing
import model_builder
import experiment_manager
import train
import evaluate
import predict_test

for module in [
    config,
    download_dataset,
    split_dataset,
    data_preprocessing,
    model_builder,
    experiment_manager,
    train,
    evaluate,
    predict_test,
]:
    importlib.reload(module)


## 3. Pipeline options

In [16]:
RUN_DOWNLOAD = True # Enable or disable dataset download.
RUN_SPLIT = True # Enable or disable dataset splitting.
RUN_PREPROCESSING = True # Enable or disable preprocessing.
RUN_MODEL_BUILD = True # Enable or disable model building.
RUN_TRAIN = True # Enable or disable model training.
RUN_EVALUATE = True # Enable or disable experiment evaluation.
RUN_UPDATE_BEST = True # Enable or disable best-model tracking after each experiment.
RUN_PREDICT = True # Enable or disable prediction.
USE_BEST_MODEL = True # Use the best checkpoint from the current experiment instead of the final saved model for evaluation and prediction.


## 4. Download and dataset split

In [17]:
if RUN_DOWNLOAD:
    download_dataset.download_dataset()
else:
    print("Download skipped.")

if RUN_SPLIT:
    split_dataset.split_dataset()
else:
    print("Dataset split skipped.")


Dataset saved to : data\raw
DATASET SPLIT
Source     : data\raw\Garbage classification\Garbage classification
Train      : data\train
Validation : data\validation
Test       : data\test

DATASET SPLIT SUMMARY
Class               Train     Validation      Test     Total
------------------------------------------------------------------------
cardboard             282             60        61       403
glass                 350             75        76       501
metal                 287             61        62       410
paper                 415             89        90       594
plastic               337             72        73       482
trash                  95             21        21       137
------------------------------------------------------------------------
TOTAL                1766            378       383      2527

Requested percentages :
- Training : 70.00 %
- Validation   : 15.00 %
- Test         : 15.00 %

Actual percentages after integer rounding :
- Training : 69.

## 5. Data loading and preprocessing

In [18]:
if RUN_PREPROCESSING:
    (
        train_dataset,
        validation_dataset,
        test_dataset,
        class_names,
    ) = data_preprocessing.get_datasets()
else:
    raise RuntimeError(
        "RUN_PREPROCESSING must be True."
    )



DATASET INFORMATION
Image size : 224 x 224
Batch size : 64
Number of classes : 6

Detected classes :
- 1. cardboard
- 2. glass
- 3. metal
- 4. paper
- 5. plastic
- 6. trash

Number of batches :
- Training : 28
- Validation   : 6
- Test         : 6


## 6. Model configuration

In [19]:
MODEL_NAME = "resnet50"  # "cnn" or "resnet50"

if MODEL_NAME == "cnn":
    MODEL_CONFIG = {
        "conv_filters": [16, 32, 64],
        "kernel_size": 3,
        "activation": "relu",
        "padding": "same",
        "pool_size": 2,
        "use_batch_normalization": True,
        "dense_units": 64,
        "dropout_rate": 0.3,
        "optimizer": "adam",
        "learning_rate": 5e-4,
        "loss": "sparse_categorical_crossentropy",
        "random_flip": True,
        "random_rotation": 0.03,
        "random_zoom": 0.03,
        "random_contrast": 0.03,
    }

elif MODEL_NAME == "resnet50":
    MODEL_CONFIG = {
        "weights": "imagenet",
        "trainable_base": False,
        "dense_units": 64,
        "dropout_rate": 0.40,
        "activation": "relu",
        "optimizer": "adam",
        "learning_rate": 1e-4,
        "loss": "sparse_categorical_crossentropy",
        "random_flip": True,
        "random_rotation": 0.10,
        "random_zoom": 0.10,
        "random_contrast": 0.10,
    }

else:
    raise ValueError(
        "MODEL_NAME doit être 'cnn' ou 'resnet50'."
    )

TRAINING_CONFIG = {
    "epochs": 25,
    "early_stopping_patience": 13,
    "reduce_lr_patience": 3,
    "reduce_lr_factor": 0.30,
    "minimum_learning_rate": 1e-7,
}


## 7. Model building

In [20]:
model = None

if RUN_MODEL_BUILD:
    model = model_builder.build_model(
        model_name=MODEL_NAME,
        number_of_classes=len(class_names),
        model_config=MODEL_CONFIG,
    )

    model.summary()
else:
    print("Model building skipped.")


Model: "resnet50_configurable"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ image_entree (InputLayer)            │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ augmentation_donnees (Sequential)    │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ pretraitement_resnet                 │ (None, 224, 224, 3)         │               0 │
│ (PretraitementResNet50)              │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ resnet50 (Functional)                │ (None, 7, 7, 2048)          │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling               │ (None, 2048)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_classification (Dense)         │ (None, 64)                  │         131,136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_classification (Dropout)     │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ predictions (Dense)                  │ (None, 6)                   │             390 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 23,719,238 (90.48 MB)

 Trainable params: 131,526 (513.77 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

## 8. Experiment creation

In [21]:
experiment_directory = (
    experiment_manager.create_experiment_directory(
        model_name=MODEL_NAME,
    )
)

experiment_manager.save_experiment_metadata(
    experiment_directory=experiment_directory,
    model_name=MODEL_NAME,
    class_names=class_names,
)



EXPERIMENT CREATED
Name : resnet_20260711_175929
Path : experiments\resnet_20260711_175929


## 9. Training

In [ ]:
history = None

if RUN_TRAIN:
    model, history = train.train_model(
        model=model,
        train_dataset=train_dataset,
        validation_dataset=validation_dataset,
        experiment_directory=experiment_directory,
        model_config=MODEL_CONFIG,
        training_config=TRAINING_CONFIG,
    )
else:
    print("Training skipped.")


File saved : experiments\resnet_20260711_175929\model_config.json
File saved : experiments\resnet_20260711_175929\training_config.json


File saved : experiments\resnet_20260711_175929\model_summary.txt

MODEL TRAINING
Maximum number of epochs : 25
Epoch 1/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 104s 3s/step - accuracy: 0.2899 - loss: 1.7969 - val_accuracy: 0.5132 - val_loss: 1.3367 - learning_rate: 1.0000e-04
Epoch 2/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 157s 6s/step - accuracy: 0.5187 - loss: 1.2728 - val_accuracy: 0.6455 - val_loss: 1.0153 - learning_rate: 1.0000e-04
Epoch 3/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 140s 5s/step - accuracy: 0.6234 - loss: 1.0391 - val_accuracy: 0.6878 - val_loss: 0.8358 - learning_rate: 1.0000e-04
Epoch 4/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 153s 6s/step - accuracy: 0.7010 - loss: 0.8625 - val_accuracy: 0.7169 - val_loss: 0.7370 - learning_rate: 1.0000e-04
Epoch 5/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 100s 4s/step - accuracy: 0.7191 - loss: 0.7708 - val_accuracy: 0.7460 - val_loss: 0.6794 - learning_rate: 1.0000e-04
Epoch 6/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 133s 5s/step - accuracy: 0.7582 - loss: 0.6997 - val_accuracy: 0.7566 - val

## 10. Evaluation

In [ ]:
evaluation_results = None

if RUN_EVALUATE:
    evaluation_results = evaluate.evaluate_experiment(
        experiment_directory=experiment_directory,
        test_dataset=test_dataset,
        class_names=class_names,
        use_best_model=USE_BEST_MODEL,
    )
else:
    print("Evaluation skipped.")


## 11. Best-model comparison

In [ ]:
best_model_result = None

if RUN_UPDATE_BEST:
    best_model_result = (
        experiment_manager.update_best_model(
            experiment_directory=experiment_directory,
            model_name=MODEL_NAME,
            history=history,
            evaluation_results=evaluation_results,
        )
    )
else:
    print("Best-model comparison skipped.")


## 12. Training curves and confusion matrix

In [ ]:
for filename in [
    "accuracy.png",
    "loss.png",
    "confusion_matrix.png",
]:
    artifact_path = (
        experiment_directory / filename
    )

    if artifact_path.exists():
        display(
            IPyImage(
                filename=str(artifact_path),
                width=650
            )
        )
    else:
        print(
            f"File not found : {filename}"
        )


## 13. Single-image prediction

In [ ]:
SELECTED_CLASS = "plastic"
IMAGE_NAME = "plastic107.jpg"
SHOW_AVAILABLE_IMAGES = False # Display a few image names to help select a test image.

prediction_result = None

if RUN_PREDICT:
    if SHOW_AVAILABLE_IMAGES:
        predict_test.display_available_images(
            class_name=SELECTED_CLASS,
            class_names=class_names,
            maximum=20,
        )

    selected_image_path = (
        predict_test.build_test_image_path(
            class_name=SELECTED_CLASS,
            image_name=IMAGE_NAME,
            class_names=class_names,
        )
    )

    predict_test.display_selected_image(
        image_path=selected_image_path,
    )

    prediction_result = (
        predict_test.predict_experiment_image(
            experiment_directory=experiment_directory,
            image_path=selected_image_path,
            class_names=class_names,
            use_best_model=USE_BEST_MODEL,
        )
    )
else:
    print("Prédiction ignorée.")


# END